# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [8]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
import os

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [9]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_ = X.copy()
        X_['timestamp'] = pd.to_datetime(X_['timestamp'])
        X_['hour'] = X_['timestamp'].dt.hour
        X_['dayofweek'] = X_['timestamp'].dt.dayofweek
        X_.drop('timestamp', axis=1, inplace=True)
        return X_

In [10]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    
    def __init__(self, target_name='dayofweek'):
        self.target_name = target_name
        
    def fit(self, X, y=None):
        X_ = X.copy()
        self.object_cols = list(X_.select_dtypes(include='object').columns)
        if self.target_name in self.object_cols:
            self.object_cols.remove(self.target_name)
        
        self.encoder = OneHotEncoder(sparse_output=False)
        self.encoder.fit(X_[self.object_cols])
        
        return self
            
    
    def transform(self, X, y=None):
        X_ = X.copy()
        target_col = pd.Series(X_[self.target_name])
        X_ = X_.drop(self.target_name, axis=1)
        
        if not self.object_cols:
            return X_, target_col
        
        one_hot_encoded = self.encoder.transform(X_[self.object_cols])
        one_hot_df = pd.DataFrame(one_hot_encoded, columns=self.encoder.get_feature_names_out(self.object_cols), index=X_.index)
        X_encoded = pd.concat([X_.drop(self.object_cols, axis=1), one_hot_df], axis=1)
        
        return X_encoded, target_col

In [11]:
class TrainValidationTest(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y):
        X_ = X.copy()
        y_ = y.copy()
        
        X_train, X_test, y_train, y_test = train_test_split(X_, y_, test_size=0.2, stratify=y_, random_state=21)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=21)
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [12]:
class ModelSelection:
    
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        best_score = -1
        best_model_name = ""
        
        for i, gs in enumerate(self.grids):
            model_name = self.grid_dict[i]
            print(f'Estimator: {model_name}')
        
            with tqdm(total=1, desc=model_name) as pbar:
                gs.fit(X_train, y_train)
                pbar.update(1)
            
            y_pred = gs.predict(X_valid)
            valid_score = accuracy_score(y_valid, y_pred)
            
            self.results.append({
                'model': model_name,
                'params': gs.best_params_,
                'valid_score': valid_score
            })
            
            print(f'Best params: {gs.best_params_}')
            print(f'Best training accuracy: {gs.best_score_:.3f}')
            print(f'Validation set accuracy score for best params: {valid_score:.3f}')
            
            if valid_score > best_score:
                best_score = valid_score
                best_model_name = model_name
        
        print(f'Classifier with best validation set accuracy: {best_model_name}')
        return best_model_name
    
    def best_results(self):
        return pd.DataFrame(self.results)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [13]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
    
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        print(f'Accuracy of the final model is {acc:.4f}')
        return acc
    
    def save_model(self, path):
        if os.path.exists(os.path.dirname(path)):
            joblib.dump(self.estimator, path)
            print("Model saved successfully.")
        else:
            print("Incorrect path for saving the model.")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [14]:
df = pd.read_csv('data/checker_submits.csv')

preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
X, y = preprocessing.fit_transform(df)

In [15]:
X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest().transform(X, y)

In [16]:
svm_params = [{
    'kernel': ('linear', 'rbf', 'sigmoid'),
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ('balanced', None),
    'random_state': [21],
    'probability': [True]
}]

tree_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 15, 20, 25],
    'class_weight': [None, 'balanced'],
    'random_state': [21]
}

rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced'],
    'random_state': [21]
}

In [17]:
gs_svm = GridSearchCV(SVC(), svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(DecisionTreeClassifier(), tree_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(RandomForestClassifier(), rf_params, scoring='accuracy', cv=2, n_jobs=-1)

In [18]:
grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {
    0: 'SVM',
    1: 'Decision Tree',
    2: 'Random Forest'
}

In [19]:
model_selector = ModelSelection(grids, grid_dict)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)
results_df = model_selector.best_results()

Estimator: SVM


SVM:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855
Estimator: Decision Tree


Decision Tree:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.864
Estimator: Random Forest


Random Forest:   0%|          | 0/1 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'n_estimators': 200, 'random_state': 21}
Best training accuracy: 0.858
Validation set accuracy score for best params: 0.884
Classifier with best validation set accuracy: Random Forest


In [20]:
final_estimator = RandomForestClassifier(class_weight=None, criterion='gini', max_depth=None, n_estimators=200, random_state=21)
final = Finalize(final_estimator)

results = final.final_score(X_train, y_train, X_test, y_test)
final.save_model(f'../data/name_of_the_model_{results}.sav')

Accuracy of the final model is 0.9112
Incorrect path for saving the model.
